# DiffMaskGIT: Enhancing Masked Generative Image Transformers
## via Diffusion-Guided Training and Hybrid Confidence-Aware Sampling

**CS7150 Deep Learning — Progress Report 3 Experiments**

Authors: Zhipeng Ling, Zichen Tian

This notebook implements and evaluates four methods:
1. **VQ-VAE** — Reconstruction baseline (rFID ceiling)
2. **MaskGIT** — Standard baseline (learned pos-emb + CE loss + Gumbel-Max)
3. **MaskGIT + 2D Axial RoPE** — Isolates positional encoding improvement
4. **DiffMaskGIT (Ours)** — Full method (diffusion loss + RoPE + hybrid sampling)

**Hardware**: Google Colab T4 GPU (15GB VRAM)
**Dataset**: ImageNette (10 ImageNet classes, 128×128)
**Runtime**: ~3-4 hours total

## Section 0: Setup & Configuration

In [ ]:
# Cell 0.1: Mount Google Drive & Install Dependencies
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
for pkg in ['diffusers', 'transformers', 'accelerate', 'einops', 'clean-fid', 'scipy', 'seaborn']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import os
PROJECT_DIR = '/content/drive/MyDrive/DiffMaskGIT'
for d in ['checkpoints', 'figures', 'data', 'data/latents']:
    os.makedirs(f'{PROJECT_DIR}/{d}', exist_ok=True)
print(f"Project directory: {PROJECT_DIR}")
print("Setup complete!")

In [ ]:
# Cell 0.2: Imports & Configuration
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torch.amp import autocast, GradScaler
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150
import seaborn as sns
import math, os, json, time, warnings
from pathlib import Path
from tqdm.auto import tqdm
from collections import defaultdict
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Hyperparameters
config = {
    # Data
    'image_size': 128,
    'latent_size': 16,      # 128 / 8 (VAE downsample factor)
    'latent_channels': 4,
    'num_classes': 10,
    # Transformer
    'num_layers': 8,
    'num_heads': 8,
    'embed_dim': 256,
    'mlp_ratio': 4,
    'dropout': 0.1,
    # VQ
    'codebook_size': 1024,
    'codebook_dim': 4,
    'vq_ema_decay': 0.99,
    'vq_commitment': 0.25,
    # Diffusion
    'diff_blocks': 2,
    'diff_channels': 256,
    'train_timesteps': 1000,
    'inference_timesteps': 50,
    # Sampling
    'maskgit_steps': 12,
    'gumbel_temp': 4.5,
    'tau_start': 0.5,
    'tau_end': 0.9,
    # Training
    'epochs': 100,
    'batch_size': 64,
    'lr': 1e-4,
    'weight_decay': 0.05,
    'warmup_epochs': 5,
    # Paths
    'project_dir': PROJECT_DIR,
}
print("Config loaded.")
for k, v in config.items():
    print(f"  {k}: {v}")

## Section 1: Data Preparation
Download ImageNette and pre-encode all images through frozen Stable Diffusion VAE.

In [ ]:
# Cell 1.1: Download ImageNette
import tarfile, urllib.request

DATA_DIR = f"{config['project_dir']}/data"
IMAGENETTE_DIR = f"{DATA_DIR}/imagenette2-320"

if not os.path.exists(IMAGENETTE_DIR):
    url = "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz"
    tgz_path = f"{DATA_DIR}/imagenette2-320.tgz"
    if not os.path.exists(tgz_path):
        print("Downloading ImageNette (~1.5GB)...")
        urllib.request.urlretrieve(url, tgz_path)
        print("Download complete.")
    print("Extracting...")
    with tarfile.open(tgz_path, 'r:gz') as tar:
        tar.extractall(DATA_DIR)
    print("Extraction complete.")
else:
    print("ImageNette already exists on Drive.")

# List classes
classes = sorted(os.listdir(f"{IMAGENETTE_DIR}/train"))
print(f"Classes ({len(classes)}): {classes}")
total_train = sum(len(os.listdir(f"{IMAGENETTE_DIR}/train/{c}")) for c in classes)
total_val = sum(len(os.listdir(f"{IMAGENETTE_DIR}/val/{c}")) for c in classes)
print(f"Training images: {total_train}, Validation images: {total_val}")

In [ ]:
# Cell 1.2: Dataset Class
class ImageNetteDataset(Dataset):
    def __init__(self, root, split='train', image_size=128):
        self.root = f"{root}/{split}"
        self.transform = T.Compose([
            T.Resize(image_size),
            T.CenterCrop(image_size),
            T.ToTensor(),
            T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),  # [-1, 1]
        ])
        self.samples = []
        self.class_names = sorted(os.listdir(self.root))
        for ci, cname in enumerate(self.class_names):
            cdir = os.path.join(self.root, cname)
            for fname in sorted(os.listdir(cdir)):
                if fname.lower().endswith(('.jpeg', '.jpg', '.png', '.JPEG')):
                    self.samples.append((os.path.join(cdir, fname), ci))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        from PIL import Image
        img = Image.open(path).convert('RGB')
        img = self.transform(img)
        return img, label

train_dataset = ImageNetteDataset(IMAGENETTE_DIR, 'train', config['image_size'])
val_dataset = ImageNetteDataset(IMAGENETTE_DIR, 'val', config['image_size'])
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
# Cell 1.3: Pre-encode images through frozen SD VAE
from diffusers import AutoencoderKL

LATENT_PATH_TRAIN = f"{config['project_dir']}/data/latents/train_latents.pt"
LATENT_PATH_VAL = f"{config['project_dir']}/data/latents/val_latents.pt"

# Load VAE
vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse").to(device)
vae.eval()
for p in vae.parameters():
    p.requires_grad = False

def encode_dataset(dataset, save_path, batch_size=32):
    """Encode all images to VAE latents and save."""
    if os.path.exists(save_path):
        data = torch.load(save_path, map_location='cpu')
        print(f"Loaded cached latents from {save_path}: {data['latents'].shape}")
        return data['latents'], data['labels']

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    all_latents, all_labels = [], []

    print(f"Encoding {len(dataset)} images...")
    for imgs, labels in tqdm(loader):
        imgs = imgs.to(device)
        with torch.no_grad():
            posterior = vae.encode(imgs).latent_dist
            latents = posterior.sample() * 0.18215  # SD scaling factor
        all_latents.append(latents.cpu())
        all_labels.append(labels)

    all_latents = torch.cat(all_latents, 0)
    all_labels = torch.cat(all_labels, 0)
    torch.save({'latents': all_latents, 'labels': all_labels}, save_path)
    print(f"Saved latents {all_latents.shape} to {save_path}")
    return all_latents, all_labels

train_latents, train_labels = encode_dataset(train_dataset, LATENT_PATH_TRAIN)
val_latents, val_labels = encode_dataset(val_dataset, LATENT_PATH_VAL)
print(f"Train latents: {train_latents.shape}, Val latents: {val_latents.shape}")

# Create DataLoaders for latents
# Reshape: [B, 4, 16, 16] -> [B, 256, 4] (flatten spatial dims)
def make_latent_loader(latents, labels, batch_size, shuffle=True):
    B, C, H, W = latents.shape
    flat = latents.permute(0, 2, 3, 1).reshape(B, H*W, C)  # [B, 256, 4]
    ds = TensorDataset(flat, labels)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=True, pin_memory=True)

train_loader = make_latent_loader(train_latents, train_labels, config['batch_size'])
val_loader = make_latent_loader(val_latents, val_labels, config['batch_size'], shuffle=False)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

In [ ]:
# Cell 1.4: Visualize sample images and latent representations
def decode_latents(latents_flat, vae_model):
    """Decode flat latents [B, 256, 4] -> images [B, 3, 128, 128]."""
    B = latents_flat.shape[0]
    lat = latents_flat.reshape(B, 16, 16, 4).permute(0, 3, 1, 2)  # [B, 4, 16, 16]
    lat = lat / 0.18215  # undo scaling
    with torch.no_grad():
        imgs = vae_model.decode(lat.to(device)).sample
    return imgs.clamp(-1, 1).cpu()

# Show some training images
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
fig.suptitle('Sample Training Images and Their VAE Latent Representations', fontsize=12)

for i in range(8):
    img, label = train_dataset[i * 100]
    axes[0, i].imshow(img.permute(1, 2, 0).numpy() * 0.5 + 0.5)
    axes[0, i].set_title(f'Class {label}', fontsize=8)
    axes[0, i].axis('off')

    # Show latent channels
    lat = train_latents[i * 100]  # [4, 16, 16]
    axes[1, i].imshow(lat[0].numpy(), cmap='viridis')
    axes[1, i].set_title('Latent ch.0', fontsize=8)
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig(f"{config['project_dir']}/figures/01_data_samples.png", bbox_inches='tight')
plt.show()
print("Figure saved.")

## Section 2: Model Definitions
All model components: 2D Axial RoPE, Transformer, VQ Codebook, Diffusion Loss MLP, and Sampling Strategies.

In [ ]:
# Cell 2.1: 2D Axial Rotary Position Embedding (RoPE)

class AxialRoPE2D(nn.Module):
    """2D Axial Rotary Position Embedding.

    Splits head dimension in half: first half rotated by row position,
    second half rotated by column position. Parameter-free.
    """
    def __init__(self, head_dim, grid_size=16):
        super().__init__()
        half = head_dim // 2
        quarter = half // 2
        freqs = 1.0 / (10000.0 ** (torch.arange(0, quarter, dtype=torch.float32) / quarter))

        positions = torch.arange(grid_size, dtype=torch.float32)
        row_pos = positions.unsqueeze(1).repeat(1, grid_size).reshape(-1)
        col_pos = positions.unsqueeze(0).repeat(grid_size, 1).reshape(-1)

        row_angles = torch.outer(row_pos, freqs)  # [N, quarter]
        col_angles = torch.outer(col_pos, freqs)  # [N, quarter]

        self.register_buffer('row_sin', row_angles.sin())
        self.register_buffer('row_cos', row_angles.cos())
        self.register_buffer('col_sin', col_angles.sin())
        self.register_buffer('col_cos', col_angles.cos())

    def _rotate(self, x, sin, cos):
        """Apply rotary embedding to pairs of elements."""
        x1, x2 = x[..., 0::2], x[..., 1::2]
        out = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
        return out.flatten(-2)

    def forward(self, q, k):
        """Apply 2D RoPE to query and key tensors.

        Args:
            q, k: [B, heads, seq_len, head_dim]
        Returns:
            rotated q, k with same shape
        """
        half = q.shape[-1] // 2
        # Split into row and column halves
        q_row, q_col = q[..., :half], q[..., half:]
        k_row, k_col = k[..., :half], k[..., half:]

        q_row = self._rotate(q_row, self.row_sin, self.row_cos)
        q_col = self._rotate(q_col, self.col_sin, self.col_cos)
        k_row = self._rotate(k_row, self.row_sin, self.row_cos)
        k_col = self._rotate(k_col, self.col_sin, self.col_cos)

        q_out = torch.cat([q_row, q_col], dim=-1)
        k_out = torch.cat([k_row, k_col], dim=-1)
        return q_out, k_out

# Quick test
rope = AxialRoPE2D(head_dim=32, grid_size=16).to(device)
q_test = torch.randn(2, 8, 256, 32, device=device)
k_test = torch.randn(2, 8, 256, 32, device=device)
q_r, k_r = rope(q_test, k_test)
print(f"RoPE test: input {q_test.shape} -> output {q_r.shape}")
del rope, q_test, k_test, q_r, k_r

In [ ]:
# Cell 2.2: Bidirectional Transformer Backbone

class TransformerBlock(nn.Module):
    def __init__(self, dim, heads, mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.heads = heads
        self.head_dim = dim // heads
        self.scale = self.head_dim ** -0.5
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.qkv = nn.Linear(dim, 3 * dim)
        self.proj = nn.Linear(dim, dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * mlp_ratio),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * mlp_ratio, dim),
            nn.Dropout(dropout),
        )
        self.attn_drop = nn.Dropout(dropout)

    def forward(self, x, rope=None, return_attn=False):
        B, N, C = x.shape
        # Pre-norm self-attention
        h = self.norm1(x)
        qkv = self.qkv(h).reshape(B, N, 3, self.heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)  # each [B, heads, N, head_dim]

        if rope is not None:
            q, k = rope(q, k)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn_weights = attn  # save for visualization
        attn = self.attn_drop(attn)

        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        out = self.proj(out)
        x = x + out

        # Pre-norm MLP
        x = x + self.mlp(self.norm2(x))

        if return_attn:
            return x, attn_weights
        return x


class MaskGITTransformer(nn.Module):
    """Bidirectional Masked Transformer for image generation.

    Supports two modes:
    - 'ce': Cross-entropy mode (for VQ token prediction)
    - 'diffusion': Outputs conditioning z-vectors for diffusion loss
    """
    def __init__(self, config, use_rope=False, output_mode='ce'):
        super().__init__()
        dim = config['embed_dim']
        self.dim = dim
        self.output_mode = output_mode
        self.latent_channels = config['latent_channels']
        self.grid_size = config['latent_size']
        self.num_tokens = self.grid_size ** 2

        # Token embedding
        if output_mode == 'ce':
            self.token_embed = nn.Embedding(config['codebook_size'] + 1, dim)  # +1 for [MASK]
            self.mask_token_id = config['codebook_size']
        else:
            self.continuous_proj = nn.Linear(config['latent_channels'], dim)
            self.mask_token = nn.Parameter(torch.randn(dim) * 0.02)

        # Position embedding
        self.use_rope = use_rope
        if use_rope:
            self.rope = AxialRoPE2D(dim // config['num_heads'], self.grid_size)
        else:
            self.pos_embed = nn.Parameter(torch.randn(1, self.num_tokens, dim) * 0.02)

        # Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(dim, config['num_heads'], config['mlp_ratio'], config['dropout'])
            for _ in range(config['num_layers'])
        ])
        self.norm = nn.LayerNorm(dim)

        # Output head
        if output_mode == 'ce':
            self.head = nn.Linear(dim, config['codebook_size'])
        else:
            self.head = nn.Linear(dim, dim)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

    def forward(self, x, mask=None, return_attn=False):
        """
        Args:
            x: [B, N] int tokens (CE mode) or [B, N, C] continuous (diffusion mode)
            mask: [B, N] bool, True = masked positions
        Returns:
            logits [B, N, codebook] or z-vectors [B, N, dim]
        """
        if self.output_mode == 'ce':
            if mask is not None:
                x = x.clone()
                x[mask] = self.mask_token_id
            h = self.token_embed(x)
        else:
            h = self.continuous_proj(x)
            if mask is not None:
                h = torch.where(mask.unsqueeze(-1), self.mask_token.unsqueeze(0).unsqueeze(0).expand_as(h), h)

        if not self.use_rope:
            h = h + self.pos_embed

        rope = self.rope if self.use_rope else None
        attns = []
        for block in self.blocks:
            if return_attn:
                h, attn = block(h, rope, return_attn=True)
                attns.append(attn.detach().cpu())
            else:
                h = block(h, rope)

        h = self.norm(h)
        out = self.head(h)

        if return_attn:
            return out, attns
        return out

# Quick test
model_test = MaskGITTransformer(config, use_rope=True, output_mode='diffusion').to(device)
x_test = torch.randn(2, 256, 4, device=device)
mask_test = torch.rand(2, 256, device=device) > 0.5
out_test = model_test(x_test, mask_test)
n_params = sum(p.numel() for p in model_test.parameters())
print(f"Transformer test: input {x_test.shape}, mask {mask_test.shape} -> output {out_test.shape}")
print(f"Parameters: {n_params/1e6:.1f}M")
del model_test, x_test, mask_test, out_test

In [ ]:
# Cell 2.3: Vector Quantization Layer

class VectorQuantizer(nn.Module):
    """EMA-updated vector quantization codebook."""
    def __init__(self, codebook_size=1024, codebook_dim=4, ema_decay=0.99, commitment=0.25):
        super().__init__()
        self.codebook_size = codebook_size
        self.codebook_dim = codebook_dim
        self.commitment = commitment
        self.ema_decay = ema_decay

        self.embedding = nn.Embedding(codebook_size, codebook_dim)
        nn.init.uniform_(self.embedding.weight, -1.0 / codebook_size, 1.0 / codebook_size)

        self.register_buffer('ema_cluster_size', torch.zeros(codebook_size))
        self.register_buffer('ema_embed_sum', self.embedding.weight.data.clone())

    def forward(self, x):
        """
        Args:
            x: [B, N, D] continuous latent tokens
        Returns:
            quantized: [B, N, D], indices: [B, N], loss: scalar
        """
        B, N, D = x.shape
        flat = x.reshape(-1, D)  # [B*N, D]

        # Distances to codebook entries
        dists = (flat.pow(2).sum(1, keepdim=True)
                 - 2 * flat @ self.embedding.weight.t()
                 + self.embedding.weight.pow(2).sum(1, keepdim=True).t())

        indices = dists.argmin(dim=-1)  # [B*N]
        quantized = self.embedding(indices).reshape(B, N, D)

        # EMA update
        if self.training:
            encodings = F.one_hot(indices, self.codebook_size).float()
            self.ema_cluster_size.mul_(self.ema_decay).add_(encodings.sum(0), alpha=1 - self.ema_decay)
            embed_sum = flat.t() @ encodings  # [D, K]
            self.ema_embed_sum.mul_(self.ema_decay).add_(embed_sum.t(), alpha=1 - self.ema_decay)

            n = self.ema_cluster_size.sum()
            cluster_size = (self.ema_cluster_size + 1e-5) / (n + self.codebook_size * 1e-5) * n
            self.embedding.weight.data.copy_(self.ema_embed_sum / cluster_size.unsqueeze(1))

        # Losses
        commitment_loss = F.mse_loss(x, quantized.detach())
        codebook_loss = F.mse_loss(quantized, x.detach())
        loss = codebook_loss + self.commitment * commitment_loss

        # Straight-through estimator
        quantized_st = x + (quantized - x).detach()

        return quantized_st, indices.reshape(B, N), loss

    def decode_indices(self, indices):
        """Convert indices to embeddings. indices: [B, N] -> [B, N, D]"""
        return self.embedding(indices)

# Quick test
vq_test = VectorQuantizer(1024, 4).to(device)
x_test = torch.randn(2, 256, 4, device=device)
q, idx, loss = vq_test(x_test)
print(f"VQ test: input {x_test.shape} -> quantized {q.shape}, indices {idx.shape}, loss {loss.item():.4f}")
print(f"Unique codebook entries used: {idx.unique().numel()}")
del vq_test, x_test, q, idx, loss

In [ ]:
# Cell 2.4: Diffusion Loss MLP

class SinusoidalTimestepEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000.0) * torch.arange(half, device=t.device).float() / half)
        args = t.float().unsqueeze(-1) * freqs.unsqueeze(0)
        return torch.cat([args.cos(), args.sin()], dim=-1)


class AdaLNResBlock(nn.Module):
    """Residual block with Adaptive Layer Normalization."""
    def __init__(self, channels, cond_dim):
        super().__init__()
        self.norm = nn.LayerNorm(channels)
        self.linear1 = nn.Linear(channels, channels)
        self.linear2 = nn.Linear(channels, channels)
        self.act = nn.SiLU()
        # AdaLN: condition -> scale, shift
        self.ada_proj = nn.Linear(cond_dim, 2 * channels)

    def forward(self, x, cond):
        """x: [B, D], cond: [B, cond_dim]"""
        h = self.norm(x)
        scale, shift = self.ada_proj(cond).chunk(2, dim=-1)
        h = h * (1 + scale) + shift
        h = self.act(self.linear1(h))
        h = self.linear2(h)
        return x + h


class DiffusionLoss(nn.Module):
    """Lightweight denoising MLP for diffusion loss on continuous tokens.

    Following MAR: takes transformer z-vectors as conditioning, predicts noise.
    """
    def __init__(self, token_dim=4, cond_dim=256, hidden_dim=256, num_blocks=2,
                 train_timesteps=1000, inference_timesteps=50):
        super().__init__()
        self.token_dim = token_dim
        self.train_timesteps = train_timesteps
        self.inference_timesteps = inference_timesteps

        # Timestep embedding
        self.time_embed = nn.Sequential(
            SinusoidalTimestepEmbedding(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        # Condition projection (z-vector from transformer)
        self.cond_proj = nn.Linear(cond_dim, hidden_dim)

        # Input projection
        self.input_proj = nn.Linear(token_dim, hidden_dim)

        # Residual blocks
        self.blocks = nn.ModuleList([
            AdaLNResBlock(hidden_dim, hidden_dim)
            for _ in range(num_blocks)
        ])

        # Output projection (predict noise)
        self.output_proj = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, token_dim),
        )

        # DDPM noise schedule (cosine)
        betas = self._cosine_beta_schedule(train_timesteps)
        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)
        self.register_buffer('betas', betas)
        self.register_buffer('alphas_cumprod', alphas_cumprod)
        self.register_buffer('sqrt_alphas_cumprod', alphas_cumprod.sqrt())
        self.register_buffer('sqrt_one_minus_alphas_cumprod', (1 - alphas_cumprod).sqrt())

    @staticmethod
    def _cosine_beta_schedule(timesteps, s=0.008):
        steps = torch.linspace(0, timesteps, timesteps + 1)
        f = torch.cos((steps / timesteps + s) / (1 + s) * math.pi * 0.5) ** 2
        betas = 1 - f[1:] / f[:-1]
        return betas.clamp(max=0.999)

    def forward_loss(self, target_tokens, z_cond):
        """Compute diffusion training loss.

        Args:
            target_tokens: [B, N, D] ground truth continuous tokens
            z_cond: [B, N, cond_dim] conditioning vectors from transformer
        Returns:
            loss: scalar MSE loss
        """
        B, N, D = target_tokens.shape
        flat_target = target_tokens.reshape(B * N, D)
        flat_cond = z_cond.reshape(B * N, -1)

        # Sample random timesteps
        t = torch.randint(0, self.train_timesteps, (B * N,), device=flat_target.device)

        # Add noise
        noise = torch.randn_like(flat_target)
        sqrt_alpha = self.sqrt_alphas_cumprod[t].unsqueeze(-1)
        sqrt_one_minus = self.sqrt_one_minus_alphas_cumprod[t].unsqueeze(-1)
        noisy = sqrt_alpha * flat_target + sqrt_one_minus * noise

        # Predict noise
        cond = self.time_embed(t) + self.cond_proj(flat_cond)
        h = self.input_proj(noisy)
        for block in self.blocks:
            h = block(h, cond)
        pred_noise = self.output_proj(h)

        return F.mse_loss(pred_noise, noise)

    @torch.no_grad()
    def sample(self, z_cond, num_steps=None):
        """Generate tokens via reverse diffusion.

        Args:
            z_cond: [B, N, cond_dim] conditioning from transformer
        Returns:
            samples: [B, N, D] generated continuous tokens
        """
        if num_steps is None:
            num_steps = self.inference_timesteps

        B_N = z_cond.shape[0] * z_cond.shape[1]
        flat_cond = z_cond.reshape(B_N, -1)

        # Start from pure noise
        x = torch.randn(B_N, self.token_dim, device=z_cond.device)

        # DDIM-like sampling with fewer steps
        step_indices = torch.linspace(self.train_timesteps - 1, 0, num_steps, device=z_cond.device).long()

        for i, t_val in enumerate(step_indices):
            t = torch.full((B_N,), t_val, device=z_cond.device, dtype=torch.long)
            cond = self.time_embed(t) + self.cond_proj(flat_cond)
            h = self.input_proj(x)
            for block in self.blocks:
                h = block(h, cond)
            pred_noise = self.output_proj(h)

            # DDPM reverse step
            alpha_t = self.alphas_cumprod[t_val]
            alpha_prev = self.alphas_cumprod[step_indices[i + 1]] if i + 1 < num_steps else torch.tensor(1.0, device=x.device)
            beta_t = 1 - alpha_t / alpha_prev

            pred_x0 = (x - (1 - alpha_t).sqrt() * pred_noise) / alpha_t.sqrt()
            pred_x0 = pred_x0.clamp(-3, 3)

            if i + 1 < num_steps:
                noise = torch.randn_like(x)
                x = alpha_prev.sqrt() * pred_x0 + (1 - alpha_prev).sqrt() * noise
            else:
                x = pred_x0

        return x.reshape(z_cond.shape[0], z_cond.shape[1], self.token_dim)

# Quick test
diff_test = DiffusionLoss(token_dim=4, cond_dim=256, hidden_dim=256, num_blocks=2).to(device)
tokens_test = torch.randn(2, 256, 4, device=device)
z_test = torch.randn(2, 256, 256, device=device)
loss_test = diff_test.forward_loss(tokens_test, z_test)
n_params_diff = sum(p.numel() for p in diff_test.parameters())
print(f"DiffLoss test: loss = {loss_test.item():.4f}, params = {n_params_diff/1e6:.1f}M")
# Test sampling (small batch)
sample_test = diff_test.sample(z_test[:1, :16], num_steps=10)
print(f"DiffLoss sample test: {sample_test.shape}")
del diff_test, tokens_test, z_test, loss_test, sample_test

In [ ]:
# Cell 2.5: Sampling Strategies

def cosine_mask_schedule(t, T):
    """Cosine schedule: fraction of tokens to mask at step t of T."""
    return math.cos(math.pi * t / (2 * T))

def generate_halton_sequence(grid_size=16, base1=2, base2=3):
    """Generate 2D Halton quasi-random sequence for spatial diversification."""
    N = grid_size * grid_size

    def halton_1d(n, base):
        seq = np.zeros(n)
        for i in range(n):
            f, r = 1.0, 0.0
            idx = i + 1
            while idx > 0:
                f /= base
                r += f * (idx % base)
                idx //= base
            seq[i] = r
        return seq

    h1 = halton_1d(N, base1)
    h2 = halton_1d(N, base2)
    # Map to grid
    rows = (h1 * grid_size).astype(int).clip(0, grid_size - 1)
    cols = (h2 * grid_size).astype(int).clip(0, grid_size - 1)
    indices = rows * grid_size + cols
    # Ensure unique by using the halton values directly as priorities
    priority = torch.from_numpy(h1 + h2).float()
    return priority

HALTON_PRIORITY = generate_halton_sequence(config['latent_size'])
print(f"Halton priority shape: {HALTON_PRIORITY.shape}")


@torch.no_grad()
def maskgit_generate_ce(model, vq, num_tokens=256, num_steps=12,
                        temperature=4.5, sampler='gumbel', class_label=None,
                        return_trajectory=False):
    """Generate images using MaskGIT iterative decoding with VQ tokens.

    Args:
        model: MaskGITTransformer in 'ce' mode
        vq: VectorQuantizer
        sampler: 'gumbel' or 'hybrid' or 'halton'
        return_trajectory: if True, return list of token grids at each step
    """
    model.eval()
    device = next(model.parameters()).device
    B = 1

    # Start fully masked
    tokens = torch.full((B, num_tokens), model.mask_token_id, dtype=torch.long, device=device)
    is_masked = torch.ones(B, num_tokens, dtype=torch.bool, device=device)
    halton_pri = HALTON_PRIORITY.to(device)

    trajectory = []
    oscillation_record = []  # for measuring token changes

    for step in range(num_steps):
        # Predict all positions
        logits = model(tokens)  # [B, N, codebook_size]
        probs = F.softmax(logits / max(temperature * (1 - step / num_steps), 0.1), dim=-1)

        # Only consider masked positions
        if sampler == 'gumbel':
            # Gumbel-Max sampling
            gumbel = -torch.log(-torch.log(torch.rand_like(probs) + 1e-8) + 1e-8)
            sampled = (probs.log() + gumbel).argmax(dim=-1)  # [B, N]
        elif sampler == 'hybrid':
            # Hybrid confidence-aware sampling
            tau = config['tau_start'] + (config['tau_end'] - config['tau_start']) * step / max(num_steps - 1, 1)
            confidence = probs.max(dim=-1).values  # [B, N]
            # High confidence: greedy
            greedy = probs.argmax(dim=-1)  # [B, N]
            # Low confidence: stochastic
            stochastic = torch.multinomial(probs.reshape(-1, probs.shape[-1]), 1).reshape(B, num_tokens)
            high_conf = confidence > tau
            sampled = torch.where(high_conf, greedy, stochastic)
        else:  # halton
            sampled = torch.multinomial(probs.reshape(-1, probs.shape[-1]), 1).reshape(B, num_tokens)

        # Fill in masked positions
        prev_tokens = tokens.clone()
        tokens = torch.where(is_masked, sampled, tokens)

        # Record trajectory
        if return_trajectory:
            trajectory.append(tokens.clone())
        # Record changes for oscillation analysis
        oscillation_record.append(tokens.clone())

        # Determine how many to keep unmasked
        if step < num_steps - 1:
            mask_ratio = cosine_mask_schedule(step + 1, num_steps)
            num_to_mask = max(1, int(num_tokens * mask_ratio))
            num_to_keep = num_tokens - num_to_mask

            # Compute confidence for re-masking decision
            confidence = probs.max(dim=-1).values  # [B, N]

            if sampler == 'halton':
                # Halton-aware: multiply confidence by Halton priority
                score = confidence * halton_pri.unsqueeze(0)
            elif sampler == 'hybrid':
                # Combine confidence with Halton spatial diversity
                score = confidence * 0.7 + halton_pri.unsqueeze(0) * 0.3
            else:
                score = confidence

            # Keep top-k by score, re-mask the rest
            _, keep_indices = score.topk(num_to_keep, dim=-1)
            is_masked = torch.ones(B, num_tokens, dtype=torch.bool, device=device)
            is_masked.scatter_(1, keep_indices, False)
            tokens[is_masked] = model.mask_token_id
        else:
            is_masked[:] = False

    # Convert token indices to continuous latents
    latents = vq.decode_indices(tokens)  # [B, N, D]

    if return_trajectory:
        return latents, trajectory, oscillation_record
    return latents


@torch.no_grad()
def maskgit_generate_diff(model, diff_mlp, num_tokens=256, num_steps=12,
                          diff_steps=50, return_trajectory=False):
    """Generate images using DiffMaskGIT with diffusion loss + hybrid sampling."""
    model.eval()
    diff_mlp.eval()
    device = next(model.parameters()).device
    B = 1

    # Start with mask tokens for all positions
    dummy_input = torch.zeros(B, num_tokens, config['latent_channels'], device=device)
    is_masked = torch.ones(B, num_tokens, dtype=torch.bool, device=device)
    generated = torch.zeros(B, num_tokens, config['latent_channels'], device=device)
    halton_pri = HALTON_PRIORITY.to(device)

    trajectory = []
    oscillation_record = []

    for step in range(num_steps):
        # Build input: generated tokens for unmasked, zeros for masked
        inp = generated.clone()

        # Get z-vectors from transformer
        z = model(inp, mask=is_masked)  # [B, N, dim]

        # Sample continuous tokens for masked positions via diffusion
        if is_masked.any():
            masked_z = z[is_masked].unsqueeze(0)  # [1, M, dim]
            sampled_tokens = diff_mlp.sample(masked_z, num_steps=diff_steps)  # [1, M, D]
            generated[is_masked] = sampled_tokens.squeeze(0)

        if return_trajectory:
            trajectory.append(generated.clone())
        oscillation_record.append(generated.clone())

        # Confidence: use negative diffusion loss as proxy
        # For simplicity, use the L2 norm of z-vectors as confidence proxy
        confidence = z.norm(dim=-1)  # [B, N]
        confidence = confidence / confidence.max()  # normalize to [0, 1]

        if step < num_steps - 1:
            mask_ratio = cosine_mask_schedule(step + 1, num_steps)
            num_to_mask = max(1, int(num_tokens * mask_ratio))
            num_to_keep = num_tokens - num_to_mask

            # Hybrid scoring
            tau = config['tau_start'] + (config['tau_end'] - config['tau_start']) * step / max(num_steps - 1, 1)
            score = confidence * 0.7 + halton_pri.unsqueeze(0) * 0.3

            _, keep_indices = score.topk(num_to_keep, dim=-1)
            is_masked = torch.ones(B, num_tokens, dtype=torch.bool, device=device)
            is_masked.scatter_(1, keep_indices, False)
        else:
            is_masked[:] = False

    if return_trajectory:
        return generated, trajectory, oscillation_record
    return generated

print("Sampling strategies defined.")

## Section 3: Training Infrastructure

In [ ]:
# Cell 3.1: Training Loop

def train_epoch_ce(model, vq, optimizer, loader, scaler, epoch):
    """Train one epoch with cross-entropy loss on VQ tokens."""
    model.train()
    total_loss = 0
    for latents, labels in loader:
        latents = latents.to(device)  # [B, N, C]

        # Quantize to get target token indices
        with torch.no_grad():
            _, target_indices, _ = vq(latents)  # [B, N]

        # Random masking
        mask_ratio = np.random.uniform(0.5, 1.0)
        mask = torch.rand(latents.shape[0], latents.shape[1], device=device) < mask_ratio

        optimizer.zero_grad()
        with autocast('cuda'):
            logits = model(target_indices, mask=mask)  # [B, N, codebook_size]
            # CE loss only on masked positions
            loss = F.cross_entropy(
                logits[mask].reshape(-1, logits.shape[-1]),
                target_indices[mask].reshape(-1)
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)


def train_epoch_diff(model, diff_mlp, optimizer, loader, scaler, epoch):
    """Train one epoch with diffusion loss on continuous tokens."""
    model.train()
    diff_mlp.train()
    total_loss = 0

    for latents, labels in loader:
        latents = latents.to(device)  # [B, N, C]

        # Random masking
        mask_ratio = np.random.uniform(0.5, 1.0)
        mask = torch.rand(latents.shape[0], latents.shape[1], device=device) < mask_ratio

        optimizer.zero_grad()
        with autocast('cuda'):
            z = model(latents, mask=mask)  # [B, N, dim]
            # Diffusion loss on masked positions
            loss = diff_mlp.forward_loss(latents, z)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(list(model.parameters()) + list(diff_mlp.parameters()), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)


def get_cosine_schedule(optimizer, epochs, warmup_epochs, min_lr=1e-6):
    """Cosine annealing with linear warmup."""
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return max(epoch / max(warmup_epochs, 1), 1e-3)
        progress = (epoch - warmup_epochs) / max(epochs - warmup_epochs, 1)
        return max(min_lr / config['lr'], 0.5 * (1 + math.cos(math.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def train_model(model, train_fn, loader, epochs, save_name, extra_params=None, extra_model=None):
    """Full training loop with checkpointing.

    Args:
        extra_params: additional parameters to optimize (e.g., diff_mlp.parameters())
        extra_model: additional nn.Module to save/restore in checkpoints (e.g., diff_mlp)
    """
    params = list(model.parameters())
    if extra_params:
        params += list(extra_params)
    optimizer = torch.optim.AdamW(params, lr=config['lr'], weight_decay=config['weight_decay'], betas=(0.9, 0.95))
    scheduler = get_cosine_schedule(optimizer, epochs, config['warmup_epochs'])
    scaler = GradScaler('cuda')

    losses = []
    best_loss = float('inf')
    ckpt_path = f"{config['project_dir']}/checkpoints/{save_name}.pt"

    # Try to resume
    start_epoch = 0
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        if 'epoch' in ckpt:
            model.load_state_dict(ckpt['model'])
            optimizer.load_state_dict(ckpt['optimizer'])
            if 'scheduler' in ckpt:
                scheduler.load_state_dict(ckpt['scheduler'])
            if 'scaler' in ckpt:
                scaler.load_state_dict(ckpt['scaler'])
            if extra_model is not None and 'extra_model' in ckpt:
                extra_model.load_state_dict(ckpt['extra_model'])
            start_epoch = ckpt['epoch'] + 1
            losses = ckpt.get('losses', [])
            best_loss = ckpt.get('best_loss', float('inf'))
            print(f"Resuming from epoch {start_epoch}")

    for epoch in tqdm(range(start_epoch, epochs), desc=f'Training {save_name}'):
        loss = train_fn(model, optimizer, loader, scaler, epoch)
        scheduler.step()
        losses.append(loss)

        if (epoch + 1) % 10 == 0 or epoch == epochs - 1:
            print(f"  Epoch {epoch+1}/{epochs}, Loss: {loss:.4f}, LR: {scheduler.get_last_lr()[0]:.6f}")

        if loss < best_loss:
            best_loss = loss

        # Save checkpoint every 20 epochs
        if (epoch + 1) % 20 == 0 or epoch == epochs - 1:
            ckpt_data = {
                'epoch': epoch,
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(),
                'scaler': scaler.state_dict(),
                'losses': losses,
                'best_loss': best_loss,
            }
            if extra_model is not None:
                ckpt_data['extra_model'] = extra_model.state_dict()
            torch.save(ckpt_data, ckpt_path)

    return losses

print("Training infrastructure defined.")

In [ ]:
# Cell 3.2: Evaluation Metrics

@torch.no_grad()
def compute_inception_features(images, batch_size=32):
    """Extract InceptionV3 features for FID computation.

    Args:
        images: tensor [N, 3, H, W] in [-1, 1] range
    Returns:
        features: [N, 2048] numpy array
    """
    from torchvision.models import inception_v3, Inception_V3_Weights
    inception = inception_v3(weights=Inception_V3_Weights.DEFAULT, transform_input=False).to(device)
    inception.eval()
    inception.fc = nn.Identity()  # remove final FC layer

    resize = T.Resize((299, 299), antialias=True)
    normalize = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

    all_features = []
    for i in range(0, len(images), batch_size):
        batch = images[i:i+batch_size].to(device)
        batch = (batch + 1) / 2  # [-1,1] -> [0,1]
        batch = normalize(resize(batch))
        feats = inception(batch)
        all_features.append(feats.detach().cpu().numpy())

    del inception
    torch.cuda.empty_cache()
    return np.concatenate(all_features, axis=0)


def compute_fid(real_features, fake_features):
    """Compute FID between two sets of InceptionV3 features."""
    from scipy.linalg import sqrtm

    mu1, sigma1 = real_features.mean(0), np.cov(real_features, rowvar=False)
    mu2, sigma2 = fake_features.mean(0), np.cov(fake_features, rowvar=False)

    diff = mu1 - mu2
    covmean, _ = sqrtm(sigma1 @ sigma2, disp=False)
    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fid = diff @ diff + np.trace(sigma1 + sigma2 - 2 * covmean)
    return float(fid)


def compute_inception_score(images, batch_size=32, splits=10):
    """Compute Inception Score."""
    from torchvision.models import inception_v3, Inception_V3_Weights

    inception = inception_v3(weights=Inception_V3_Weights.DEFAULT, transform_input=False).to(device)
    inception.eval()

    resize = T.Resize((299, 299), antialias=True)
    normalize = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

    all_preds = []
    for i in range(0, len(images), batch_size):
        batch = images[i:i+batch_size].to(device)
        batch = (batch + 1) / 2
        batch = normalize(resize(batch))
        preds = F.softmax(inception(batch), dim=1)
        all_preds.append(preds.detach().cpu().numpy())

    del inception
    torch.cuda.empty_cache()
    preds = np.concatenate(all_preds, axis=0)

    scores = []
    for k in range(splits):
        part = preds[k * len(preds) // splits: (k + 1) * len(preds) // splits]
        kl = part * (np.log(part + 1e-10) - np.log(part.mean(0, keepdims=True) + 1e-10))
        scores.append(np.exp(kl.sum(1).mean()))

    return float(np.mean(scores)), float(np.std(scores))


def compute_psnr_ssim(img1, img2):
    """Compute PSNR and SSIM between two images [C, H, W] in [-1, 1]."""
    img1 = ((img1 + 1) / 2).clamp(0, 1).numpy()
    img2 = ((img2 + 1) / 2).clamp(0, 1).numpy()
    mse = ((img1 - img2) ** 2).mean()
    psnr = -10 * np.log10(mse + 1e-10)

    # Simple SSIM (per channel, averaged)
    from scipy.ndimage import uniform_filter
    ssim_vals = []
    for c in range(3):
        a, b = img1[c], img2[c]
        mu_a = uniform_filter(a, size=11)
        mu_b = uniform_filter(b, size=11)
        sig_a = uniform_filter(a**2, size=11) - mu_a**2
        sig_b = uniform_filter(b**2, size=11) - mu_b**2
        sig_ab = uniform_filter(a*b, size=11) - mu_a*mu_b
        C1, C2 = 0.01**2, 0.03**2
        num = (2*mu_a*mu_b + C1) * (2*sig_ab + C2)
        den = (mu_a**2 + mu_b**2 + C1) * (sig_a + sig_b + C2)
        ssim_vals.append((num / den).mean())
    return float(psnr), float(np.mean(ssim_vals))

print("Metrics defined.")

## Section 4: Experiment 1 — VQ-VAE Baseline
Train a VQ codebook on the continuous VAE latents to establish the reconstruction quality ceiling (rFID).

In [ ]:
# Cell 4.1: Train VQ Codebook
vq_model = VectorQuantizer(
    codebook_size=config['codebook_size'],
    codebook_dim=config['codebook_dim'],
    ema_decay=config['vq_ema_decay'],
    commitment=config['vq_commitment']
).to(device)

vq_optimizer = torch.optim.Adam(vq_model.parameters(), lr=1e-3)
vq_losses = []

print("Training VQ codebook...")
for epoch in tqdm(range(50), desc='VQ Training'):
    epoch_loss = 0
    vq_model.train()
    for latents, _ in train_loader:
        latents = latents.to(device)
        quantized, indices, loss = vq_model(latents)
        vq_optimizer.zero_grad()
        loss.backward()
        vq_optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(train_loader)
    vq_losses.append(avg_loss)
    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1}/50, Loss: {avg_loss:.4f}")

torch.save(vq_model.state_dict(), f"{config['project_dir']}/checkpoints/vq_model.pt")
print(f"VQ training complete. Final loss: {vq_losses[-1]:.4f}")

# Check codebook utilization
vq_model.eval()
all_indices = []
with torch.no_grad():
    for latents, _ in train_loader:
        _, indices, _ = vq_model(latents.to(device))
        all_indices.append(indices.cpu())
all_indices = torch.cat(all_indices, 0)
utilization = all_indices.unique().numel() / config['codebook_size'] * 100
print(f"Codebook utilization: {utilization:.1f}% ({all_indices.unique().numel()}/{config['codebook_size']})")

In [ ]:
# Cell 4.2: Compute rFID (reconstruction quality ceiling)
print("Computing rFID...")

# Reconstruct all validation images through VQ
vq_model.eval()
all_recon_latents = []
all_orig_latents = []
with torch.no_grad():
    for latents, _ in val_loader:
        latents = latents.to(device)
        quantized, _, _ = vq_model(latents)
        all_recon_latents.append(quantized.cpu())
        all_orig_latents.append(latents.cpu())

all_recon_latents = torch.cat(all_recon_latents, 0)
all_orig_latents = torch.cat(all_orig_latents, 0)

# Decode latents to images
num_for_fid = min(1000, len(all_recon_latents))
print(f"Decoding {num_for_fid} images for FID...")

orig_images = []
recon_images = []
for i in tqdm(range(0, num_for_fid, 16)):
    batch_o = all_orig_latents[i:i+16]
    batch_r = all_recon_latents[i:i+16]
    orig_images.append(decode_latents(batch_o, vae))
    recon_images.append(decode_latents(batch_r, vae))

orig_images = torch.cat(orig_images, 0)
recon_images = torch.cat(recon_images, 0)

# Compute FID
print("Computing InceptionV3 features...")
orig_feats = compute_inception_features(orig_images)
recon_feats = compute_inception_features(recon_images)
rfid = compute_fid(orig_feats, recon_feats)
print(f"\n=== VQ-VAE Reconstruction FID (rFID): {rfid:.2f} ===")

# Also compute PSNR/SSIM
psnr_vals, ssim_vals = [], []
for i in range(min(100, len(orig_images))):
    p, s = compute_psnr_ssim(orig_images[i], recon_images[i])
    psnr_vals.append(p)
    ssim_vals.append(s)
print(f"Average PSNR: {np.mean(psnr_vals):.2f} dB, SSIM: {np.mean(ssim_vals):.4f}")

# Store results
results = {'vq_rfid': rfid, 'vq_psnr': np.mean(psnr_vals), 'vq_ssim': np.mean(ssim_vals)}

In [ ]:
# Cell 4.3: Visualize VQ reconstructions
fig, axes = plt.subplots(3, 8, figsize=(16, 6))
fig.suptitle(f'VQ-VAE Reconstruction (rFID = {rfid:.2f})', fontsize=14)

for i in range(8):
    # Original
    axes[0, i].imshow(orig_images[i].permute(1,2,0).numpy() * 0.5 + 0.5)
    axes[0, i].axis('off')
    if i == 0: axes[0, i].set_ylabel('Original', fontsize=10)

    # Reconstructed
    axes[1, i].imshow(recon_images[i].permute(1,2,0).numpy() * 0.5 + 0.5)
    axes[1, i].axis('off')
    if i == 0: axes[1, i].set_ylabel('VQ Recon', fontsize=10)

    # Difference (amplified)
    diff = (orig_images[i] - recon_images[i]).abs().mean(0)
    axes[2, i].imshow(diff.numpy(), cmap='hot', vmin=0, vmax=0.3)
    axes[2, i].axis('off')
    if i == 0: axes[2, i].set_ylabel('Difference', fontsize=10)

plt.tight_layout()
plt.savefig(f"{config['project_dir']}/figures/02_vq_reconstruction.png", bbox_inches='tight', dpi=150)
plt.show()
print("Figure saved.")

## Section 5: Experiment 2 — MaskGIT Baseline
Standard MaskGIT with learned position embeddings, cross-entropy loss, and Gumbel-Max sampling.

In [ ]:
# Cell 5.1-5.2: Train MaskGIT Baseline
maskgit_baseline = MaskGITTransformer(config, use_rope=False, output_mode='ce').to(device)
n_params = sum(p.numel() for p in maskgit_baseline.parameters())
print(f"MaskGIT Baseline: {n_params/1e6:.1f}M parameters")

def train_fn_baseline(model, optimizer, loader, scaler, epoch):
    return train_epoch_ce(model, vq_model, optimizer, loader, scaler, epoch)

maskgit_losses = train_model(
    maskgit_baseline, train_fn_baseline, train_loader,
    config['epochs'], 'maskgit_baseline'
)
print("MaskGIT baseline training complete.")

In [ ]:
# Cell 5.3-5.4: Generate and Evaluate MaskGIT Baseline
print("Generating samples with Gumbel-Max sampling...")

num_gen = 500  # Generate 500 images for FID
gen_latents_baseline = []
for i in tqdm(range(num_gen)):
    lat = maskgit_generate_ce(
        maskgit_baseline, vq_model,
        num_tokens=256, num_steps=config['maskgit_steps'],
        temperature=config['gumbel_temp'], sampler='gumbel'
    )
    gen_latents_baseline.append(lat)
gen_latents_baseline = torch.cat(gen_latents_baseline, 0)

# Decode to images
print("Decoding generated images...")
gen_images_baseline = []
for i in range(0, num_gen, 16):
    gen_images_baseline.append(decode_latents(gen_latents_baseline[i:i+16], vae))
gen_images_baseline = torch.cat(gen_images_baseline, 0)

# Compute FID
print("Computing FID...")
gen_feats_baseline = compute_inception_features(gen_images_baseline)
fid_baseline = compute_fid(orig_feats[:num_gen], gen_feats_baseline)
is_baseline, is_std = compute_inception_score(gen_images_baseline)

print(f"\n=== MaskGIT Baseline ===")
print(f"FID: {fid_baseline:.2f}")
print(f"IS: {is_baseline:.2f} +/- {is_std:.2f}")

results['maskgit_fid'] = fid_baseline
results['maskgit_is'] = is_baseline
results['maskgit_losses'] = maskgit_losses

# Show generated samples
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
fig.suptitle(f'MaskGIT Baseline Generated Samples (FID={fid_baseline:.1f})', fontsize=14)
for i in range(32):
    r, c = i // 8, i % 8
    axes[r, c].imshow(gen_images_baseline[i].permute(1,2,0).numpy() * 0.5 + 0.5)
    axes[r, c].axis('off')
plt.tight_layout()
plt.savefig(f"{config['project_dir']}/figures/03_maskgit_samples.png", bbox_inches='tight', dpi=150)
plt.show()

## Section 6: Experiment 3 — MaskGIT + 2D Axial RoPE
Same as baseline but with 2D Axial Rotary Position Embeddings replacing learned position embeddings.

In [ ]:
# Cell 6.1-6.2: Train MaskGIT + RoPE
maskgit_rope = MaskGITTransformer(config, use_rope=True, output_mode='ce').to(device)
n_params = sum(p.numel() for p in maskgit_rope.parameters())
print(f"MaskGIT + RoPE: {n_params/1e6:.1f}M parameters")

def train_fn_rope(model, optimizer, loader, scaler, epoch):
    return train_epoch_ce(model, vq_model, optimizer, loader, scaler, epoch)

rope_losses = train_model(
    maskgit_rope, train_fn_rope, train_loader,
    config['epochs'], 'maskgit_rope'
)
print("MaskGIT + RoPE training complete.")

In [ ]:
# Cell 6.3: Generate and Evaluate MaskGIT + RoPE
print("Generating samples with MaskGIT + RoPE...")

gen_latents_rope = []
for i in tqdm(range(num_gen)):
    lat = maskgit_generate_ce(
        maskgit_rope, vq_model,
        num_tokens=256, num_steps=config['maskgit_steps'],
        temperature=config['gumbel_temp'], sampler='gumbel'
    )
    gen_latents_rope.append(lat)
gen_latents_rope = torch.cat(gen_latents_rope, 0)

gen_images_rope = []
for i in range(0, num_gen, 16):
    gen_images_rope.append(decode_latents(gen_latents_rope[i:i+16], vae))
gen_images_rope = torch.cat(gen_images_rope, 0)

gen_feats_rope = compute_inception_features(gen_images_rope)
fid_rope = compute_fid(orig_feats[:num_gen], gen_feats_rope)
is_rope, is_rope_std = compute_inception_score(gen_images_rope)

print(f"\n=== MaskGIT + 2D RoPE ===")
print(f"FID: {fid_rope:.2f}")
print(f"IS: {is_rope:.2f} +/- {is_rope_std:.2f}")

results['rope_fid'] = fid_rope
results['rope_is'] = is_rope
results['rope_losses'] = rope_losses

In [ ]:
# Cell 6.4: Attention Map Visualization (Learned vs RoPE)
print("Visualizing attention maps...")

# Get attention maps from both models
sample_latent = train_latents[0:1].to(device)  # [1, 4, 16, 16]
sample_flat = sample_latent.permute(0, 2, 3, 1).reshape(1, 256, 4)

with torch.no_grad():
    # Quantize for CE models
    _, sample_indices, _ = vq_model(sample_flat)

    _, attns_learned = maskgit_baseline(sample_indices, return_attn=True)
    _, attns_rope = maskgit_rope(sample_indices, return_attn=True)

# Visualize attention from center token (128) at layers 1, 4, 7
query_token = 128  # center of 16x16 grid
layers_to_show = [0, 3, 7]

fig, axes = plt.subplots(2, len(layers_to_show), figsize=(12, 8))
fig.suptitle(f'Attention Maps: Query Token at Grid Position (8,8)', fontsize=14)

for col, layer_idx in enumerate(layers_to_show):
    # Learned pos embed - average over heads
    attn_l = attns_learned[layer_idx][0].mean(0)[query_token].reshape(16, 16).numpy()
    axes[0, col].imshow(attn_l, cmap='viridis')
    axes[0, col].set_title(f'Layer {layer_idx+1}', fontsize=10)
    if col == 0: axes[0, col].set_ylabel('Learned PosEmb', fontsize=10)
    axes[0, col].axis('off')

    # RoPE
    attn_r = attns_rope[layer_idx][0].mean(0)[query_token].reshape(16, 16).numpy()
    axes[1, col].imshow(attn_r, cmap='viridis')
    if col == 0: axes[1, col].set_ylabel('2D Axial RoPE', fontsize=10)
    axes[1, col].axis('off')

plt.tight_layout()
plt.savefig(f"{config['project_dir']}/figures/04_attention_maps.png", bbox_inches='tight', dpi=150)
plt.show()
print("Attention maps saved.")

## Section 7: Experiment 4 — DiffMaskGIT (Our Full Method)
Diffusion loss + 2D Axial RoPE + Hybrid Confidence-Aware Sampling with Halton re-masking.

In [ ]:
# Cell 7.1-7.2: Train DiffMaskGIT
diffmaskgit = MaskGITTransformer(config, use_rope=True, output_mode='diffusion').to(device)
diff_mlp = DiffusionLoss(
    token_dim=config['latent_channels'],
    cond_dim=config['embed_dim'],
    hidden_dim=config['diff_channels'],
    num_blocks=config['diff_blocks'],
    train_timesteps=config['train_timesteps'],
    inference_timesteps=config['inference_timesteps']
).to(device)

n_total = sum(p.numel() for p in diffmaskgit.parameters()) + sum(p.numel() for p in diff_mlp.parameters())
print(f"DiffMaskGIT total: {n_total/1e6:.1f}M parameters")

def train_fn_diff_wrapper(model, optimizer, loader, scaler, epoch):
    return train_epoch_diff(model, diff_mlp, optimizer, loader, scaler, epoch)

diff_losses = train_model(
    diffmaskgit, train_fn_diff_wrapper, train_loader,
    config['epochs'], 'diffmaskgit',
    extra_params=diff_mlp.parameters(),
    extra_model=diff_mlp
)
print("DiffMaskGIT training complete.")

In [ ]:
# Cell 7.3-7.4: Generate and Evaluate DiffMaskGIT
print("Generating samples with DiffMaskGIT (hybrid sampling)...")

gen_latents_diff = []
for i in tqdm(range(num_gen)):
    lat = maskgit_generate_diff(
        diffmaskgit, diff_mlp,
        num_tokens=256, num_steps=config['maskgit_steps'],
        diff_steps=config['inference_timesteps']
    )
    gen_latents_diff.append(lat)
gen_latents_diff = torch.cat(gen_latents_diff, 0)

gen_images_diff = []
for i in range(0, num_gen, 16):
    gen_images_diff.append(decode_latents(gen_latents_diff[i:i+16], vae))
gen_images_diff = torch.cat(gen_images_diff, 0)

gen_feats_diff = compute_inception_features(gen_images_diff)
fid_diff = compute_fid(orig_feats[:num_gen], gen_feats_diff)
is_diff, is_diff_std = compute_inception_score(gen_images_diff)

print(f"\n=== DiffMaskGIT (Ours) ===")
print(f"FID: {fid_diff:.2f}")
print(f"IS: {is_diff:.2f} +/- {is_diff_std:.2f}")

results['diff_fid'] = fid_diff
results['diff_is'] = is_diff
results['diff_losses'] = diff_losses

# Show generated samples
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
fig.suptitle(f'DiffMaskGIT Generated Samples (FID={fid_diff:.1f})', fontsize=14)
for i in range(32):
    r, c = i // 8, i % 8
    axes[r, c].imshow(gen_images_diff[i].permute(1,2,0).numpy() * 0.5 + 0.5)
    axes[r, c].axis('off')
plt.tight_layout()
plt.savefig(f"{config['project_dir']}/figures/05_diffmaskgit_samples.png", bbox_inches='tight', dpi=150)
plt.show()

## Section 8: Ablation Studies
1. FID vs decoding steps for different samplers
2. Token oscillation heatmap
3. Tau sensitivity analysis
4. Image completion demo

In [ ]:
# Cell 8.1: FID vs Decoding Steps
print("Ablation: FID vs decoding steps...")

step_counts = [4, 8, 12, 16, 24]
sampler_types = ['gumbel', 'halton', 'hybrid']
fid_vs_steps = {s: [] for s in sampler_types}

num_ablation = 200  # Fewer samples for speed

for sampler in sampler_types:
    print(f"  Sampler: {sampler}")
    for n_steps in step_counts:
        gen_lats = []
        for i in range(num_ablation):
            lat = maskgit_generate_ce(
                maskgit_rope, vq_model,
                num_tokens=256, num_steps=n_steps,
                temperature=config['gumbel_temp'], sampler=sampler
            )
            gen_lats.append(lat)
        gen_lats = torch.cat(gen_lats, 0)

        gen_imgs = []
        for i in range(0, num_ablation, 16):
            gen_imgs.append(decode_latents(gen_lats[i:i+16], vae))
        gen_imgs = torch.cat(gen_imgs, 0)

        feats = compute_inception_features(gen_imgs)
        fid = compute_fid(orig_feats[:num_ablation], feats)
        fid_vs_steps[sampler].append(fid)
        print(f"    Steps={n_steps}: FID={fid:.2f}")

results['fid_vs_steps'] = fid_vs_steps
results['step_counts'] = step_counts

In [ ]:
# Cell 8.2: Token Oscillation Analysis
print("Measuring token oscillation...")

# Generate with trajectory recording
_, traj_gumbel, osc_gumbel = maskgit_generate_ce(
    maskgit_rope, vq_model,
    num_tokens=256, num_steps=16,
    temperature=config['gumbel_temp'], sampler='gumbel',
    return_trajectory=True
)

_, traj_hybrid, osc_hybrid = maskgit_generate_ce(
    maskgit_rope, vq_model,
    num_tokens=256, num_steps=16,
    temperature=config['gumbel_temp'], sampler='hybrid',
    return_trajectory=True
)

def compute_oscillation_map(records, grid_size=16):
    """Count token changes between consecutive steps."""
    osc = np.zeros(grid_size * grid_size)
    for i in range(1, len(records)):
        changes = (records[i][0] != records[i-1][0]).cpu().numpy().astype(float)
        osc += changes
    return osc.reshape(grid_size, grid_size)

osc_map_gumbel = compute_oscillation_map(osc_gumbel)
osc_map_hybrid = compute_oscillation_map(osc_hybrid)

print(f"Gumbel: avg oscillations per position = {osc_map_gumbel.mean():.2f}")
print(f"Hybrid: avg oscillations per position = {osc_map_hybrid.mean():.2f}")
print(f"Reduction: {(1 - osc_map_hybrid.mean()/max(osc_map_gumbel.mean(), 1e-8))*100:.1f}%")

results['osc_gumbel_avg'] = float(osc_map_gumbel.mean())
results['osc_hybrid_avg'] = float(osc_map_hybrid.mean())

In [ ]:
# Cell 8.3: Tau Sensitivity Analysis
print("Tau sensitivity analysis...")

tau_configs = {
    'fixed_0.5': (0.5, 0.5),
    'fixed_0.7': (0.7, 0.7),
    'fixed_0.9': (0.9, 0.9),
    'annealed_0.5_0.9': (0.5, 0.9),  # our default
}

tau_fids = {}
for name, (tau_s, tau_e) in tau_configs.items():
    # Temporarily override config
    old_s, old_e = config['tau_start'], config['tau_end']
    config['tau_start'], config['tau_end'] = tau_s, tau_e

    gen_lats = []
    for i in range(num_ablation):
        lat = maskgit_generate_ce(
            maskgit_rope, vq_model,
            num_tokens=256, num_steps=config['maskgit_steps'],
            temperature=config['gumbel_temp'], sampler='hybrid'
        )
        gen_lats.append(lat)
    gen_lats = torch.cat(gen_lats, 0)

    gen_imgs = []
    for i in range(0, num_ablation, 16):
        gen_imgs.append(decode_latents(gen_lats[i:i+16], vae))
    gen_imgs = torch.cat(gen_imgs, 0)

    feats = compute_inception_features(gen_imgs)
    fid = compute_fid(orig_feats[:num_ablation], feats)
    tau_fids[name] = fid
    print(f"  {name}: FID = {fid:.2f}")

    config['tau_start'], config['tau_end'] = old_s, old_e

results['tau_fids'] = tau_fids

In [ ]:
# Cell 8.4: Image Completion Demo
print("Image completion demo...")

def image_completion(model, vq, latent, mask_type='right_half', num_steps=12, sampler='hybrid'):
    """Complete a partially masked image.

    Args:
        latent: [1, 256, 4] flat latent of a real image
        mask_type: 'right_half', 'bottom_half', 'random_50'
    """
    model.eval()
    B, N, C = latent.shape
    grid = config['latent_size']

    # Create mask
    is_masked = torch.zeros(B, N, dtype=torch.bool, device=device)
    if mask_type == 'right_half':
        for r in range(grid):
            for c in range(grid // 2, grid):
                is_masked[0, r * grid + c] = True
    elif mask_type == 'bottom_half':
        for r in range(grid // 2, grid):
            for c in range(grid):
                is_masked[0, r * grid + c] = True
    elif mask_type == 'random_50':
        idx = torch.randperm(N)[:N//2]
        is_masked[0, idx] = True

    # Quantize original
    with torch.no_grad():
        _, orig_indices, _ = vq(latent)

    tokens = orig_indices.clone()
    tokens[is_masked] = model.mask_token_id

    # Iterative completion
    halton_pri = HALTON_PRIORITY.to(device)
    remaining = is_masked.clone()

    for step in range(num_steps):
        logits = model(tokens)
        probs = F.softmax(logits, dim=-1)

        if sampler == 'hybrid':
            tau = config['tau_start'] + (config['tau_end'] - config['tau_start']) * step / max(num_steps-1, 1)
            confidence = probs.max(dim=-1).values
            greedy = probs.argmax(dim=-1)
            stochastic = torch.multinomial(probs.reshape(-1, probs.shape[-1]), 1).reshape(B, N)
            sampled = torch.where(confidence > tau, greedy, stochastic)
        else:
            gumbel = -torch.log(-torch.log(torch.rand_like(probs) + 1e-8) + 1e-8)
            sampled = (probs.log() + gumbel).argmax(dim=-1)
            confidence = probs.max(dim=-1).values

        tokens = torch.where(remaining, sampled, tokens)

        if step < num_steps - 1:
            mask_ratio = cosine_mask_schedule(step + 1, num_steps)
            num_still_masked = max(1, int(remaining.sum().item() * mask_ratio))

            score = confidence.clone()
            score[~remaining] = float('inf')  # don't re-mask known tokens
            _, worst = score.topk(num_still_masked, dim=-1, largest=False)
            remaining = torch.zeros_like(remaining)
            remaining.scatter_(1, worst, True)
            tokens[remaining] = model.mask_token_id

    completed_latent = vq.decode_indices(tokens)
    return completed_latent, is_masked

# Run completion on several images
completion_results = []
for idx in [0, 50, 100, 150]:
    lat = val_latents[idx:idx+1].permute(0, 2, 3, 1).reshape(1, 256, 4).to(device)
    for mask_type in ['right_half', 'bottom_half', 'random_50']:
        completed, mask = image_completion(maskgit_rope, vq_model, lat, mask_type, sampler='hybrid')

        orig_img = decode_latents(lat.cpu(), vae)[0]
        comp_img = decode_latents(completed.cpu(), vae)[0]

        # Create masked visualization
        masked_vis = orig_img.clone()
        mask_2d = mask[0].cpu().reshape(16, 16)
        for r in range(16):
            for c in range(16):
                if mask_2d[r, c]:
                    masked_vis[:, r*8:(r+1)*8, c*8:(c+1)*8] = 0.5  # gray

        completion_results.append({
            'orig': orig_img, 'masked': masked_vis, 'completed': comp_img,
            'mask_type': mask_type, 'idx': idx
        })

print(f"Completed {len(completion_results)} image completions.")

## Section 9: Visualizations & Results
Generate all figures for the report.

In [ ]:
# Cell 9.1: 4-Method Generated Samples Comparison
fig, axes = plt.subplots(4, 8, figsize=(16, 9))

titles = [
    f'MaskGIT (FID={results["maskgit_fid"]:.1f})',
    f'MaskGIT+RoPE (FID={results["rope_fid"]:.1f})',
    f'DiffMaskGIT (FID={results["diff_fid"]:.1f})',
    'Real Images'
]
image_sets = [gen_images_baseline, gen_images_rope, gen_images_diff, orig_images]

for row in range(4):
    for col in range(8):
        img = image_sets[row][col].permute(1,2,0).numpy() * 0.5 + 0.5
        axes[row, col].imshow(img.clip(0, 1))
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(titles[row], fontsize=9, rotation=0, labelpad=100, va='center')

fig.suptitle('Generated Samples Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f"{config['project_dir']}/figures/06_samples_comparison.png", bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Cell 9.2: FID Bar Chart
methods = ['VQ-VAE\n(rFID)', 'MaskGIT', 'MaskGIT\n+RoPE', 'DiffMaskGIT\n(Ours)']
fids = [results['vq_rfid'], results['maskgit_fid'], results['rope_fid'], results['diff_fid']]
colors = ['#95a5a6', '#3498db', '#2ecc71', '#e74c3c']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(methods, fids, color=colors, edgecolor='black', linewidth=0.5)

for bar, fid in zip(bars, fids):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{fid:.1f}', va='center', fontsize=12, fontweight='bold')

ax.set_xlabel('FID (lower is better)', fontsize=12)
ax.set_title('FID Comparison Across Methods', fontsize=14)
ax.invert_yaxis()
ax.set_xlim(0, max(fids) * 1.2)
sns.despine()
plt.tight_layout()
plt.savefig(f"{config['project_dir']}/figures/07_fid_comparison.png", bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Cell 9.3: FID vs Decoding Steps
fig, ax = plt.subplots(figsize=(8, 5))

markers = {'gumbel': 's', 'halton': '^', 'hybrid': 'D'}
colors_line = {'gumbel': '#3498db', 'halton': '#e74c3c', 'hybrid': '#2ecc71'}
labels = {'gumbel': 'Confidence (Gumbel-Max)', 'halton': 'Halton Scheduler', 'hybrid': 'Hybrid (Ours)'}

for sampler in sampler_types:
    ax.plot(results['step_counts'], results['fid_vs_steps'][sampler],
            marker=markers[sampler], color=colors_line[sampler],
            label=labels[sampler], linewidth=2, markersize=8)

ax.set_xlabel('Decoding Steps', fontsize=12)
ax.set_ylabel('FID (lower is better)', fontsize=12)
ax.set_title('FID vs Decoding Steps — Sampling Strategy Ablation', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
sns.despine()
plt.tight_layout()
plt.savefig(f"{config['project_dir']}/figures/08_fid_vs_steps.png", bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Cell 9.4: Training Loss Curves
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(maskgit_losses, label='MaskGIT (CE Loss)', color='#3498db', alpha=0.8)
ax.plot(rope_losses, label='MaskGIT+RoPE (CE Loss)', color='#2ecc71', alpha=0.8)
ax.plot(diff_losses, label='DiffMaskGIT (Diffusion Loss)', color='#e74c3c', alpha=0.8, linestyle='--')

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Training Loss Curves', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
sns.despine()
plt.tight_layout()
plt.savefig(f"{config['project_dir']}/figures/09_training_losses.png", bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Cell 9.5: Token Oscillation Heatmap
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

vmax = max(osc_map_gumbel.max(), osc_map_hybrid.max())

im1 = ax1.imshow(osc_map_gumbel, cmap='YlOrRd', vmin=0, vmax=vmax)
ax1.set_title(f'Gumbel-Max (avg={osc_map_gumbel.mean():.2f})', fontsize=12)
ax1.set_xlabel('Column')
ax1.set_ylabel('Row')
plt.colorbar(im1, ax=ax1, label='# Token Changes')

im2 = ax2.imshow(osc_map_hybrid, cmap='YlOrRd', vmin=0, vmax=vmax)
ax2.set_title(f'Hybrid Sampler (avg={osc_map_hybrid.mean():.2f})', fontsize=12)
ax2.set_xlabel('Column')
plt.colorbar(im2, ax=ax2, label='# Token Changes')

fig.suptitle('Token Oscillation Across 16 Decoding Steps', fontsize=14)
plt.tight_layout()
plt.savefig(f"{config['project_dir']}/figures/10_oscillation_heatmap.png", bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Cell 9.6: Attention Maps (already generated in Section 6)
# Re-show the saved figure
from IPython.display import Image, display
display(Image(filename=f"{config['project_dir']}/figures/04_attention_maps.png"))

In [ ]:
# Cell 9.7: Image Completion Examples
n_examples = min(4, len(completion_results) // 3)  # 4 images x 3 mask types
fig, axes = plt.subplots(n_examples, 9, figsize=(18, n_examples * 2.2))

col_titles = ['Original', 'Right Mask', 'Completed', 'Original', 'Bottom Mask', 'Completed', 'Original', 'Random Mask', 'Completed']

for row in range(n_examples):
    for mi, mask_type in enumerate(['right_half', 'bottom_half', 'random_50']):
        r_idx = row * 3 + mi
        if r_idx >= len(completion_results):
            break
        res = completion_results[r_idx]
        base_col = mi * 3

        axes[row, base_col].imshow(res['orig'].permute(1,2,0).numpy() * 0.5 + 0.5)
        axes[row, base_col+1].imshow(res['masked'].permute(1,2,0).numpy() * 0.5 + 0.5)
        axes[row, base_col+2].imshow(res['completed'].permute(1,2,0).numpy() * 0.5 + 0.5)

        for c in range(3):
            axes[row, base_col+c].axis('off')
            if row == 0:
                axes[row, base_col+c].set_title(col_titles[base_col+c], fontsize=8)

fig.suptitle('Image Completion with Hybrid Sampler', fontsize=14)
plt.tight_layout()
plt.savefig(f"{config['project_dir']}/figures/11_image_completion.png", bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Cell 9.8: Comprehensive Results Table
print("=" * 70)
print("COMPREHENSIVE RESULTS TABLE")
print("=" * 70)
print(f"{'Method':<25} {'FID':>8} {'IS':>10} {'Notes':<25}")
print("-" * 70)
print(f"{'VQ-VAE (rFID)':.<25} {results['vq_rfid']:>8.2f} {'---':>10} {'Reconstruction ceiling':<25}")
print(f"{'MaskGIT':.<25} {results['maskgit_fid']:>8.2f} {results['maskgit_is']:>8.2f}   {'Learned PosEmb + Gumbel':<25}")
print(f"{'MaskGIT + 2D RoPE':.<25} {results['rope_fid']:>8.2f} {results['rope_is']:>8.2f}   {'+ Rotary Position Emb':<25}")
print(f"{'DiffMaskGIT (Ours)':.<25} {results['diff_fid']:>8.2f} {results['diff_is']:>8.2f}   {'+ DiffLoss + Hybrid':<25}")
print("=" * 70)

# Improvements
if results['maskgit_fid'] > 0:
    imp_rope = (results['maskgit_fid'] - results['rope_fid']) / results['maskgit_fid'] * 100
    imp_diff = (results['maskgit_fid'] - results['diff_fid']) / results['maskgit_fid'] * 100
    print(f"\nRelative FID improvement over MaskGIT baseline:")
    print(f"  + 2D RoPE:      {imp_rope:+.1f}%")
    print(f"  + Full method:   {imp_diff:+.1f}%")

print(f"\nToken Oscillation (16 steps):")
print(f"  Gumbel-Max:  {results['osc_gumbel_avg']:.2f} avg changes/position")
print(f"  Hybrid:      {results['osc_hybrid_avg']:.2f} avg changes/position")
if results['osc_gumbel_avg'] > 0:
    osc_reduction = (1 - results['osc_hybrid_avg']/results['osc_gumbel_avg']) * 100
    print(f"  Reduction:   {osc_reduction:.1f}%")

print(f"\nVQ-VAE Reconstruction Quality:")
print(f"  PSNR: {results['vq_psnr']:.2f} dB")
print(f"  SSIM: {results['vq_ssim']:.4f}")

In [ ]:
# Cell 9.9: Ablation Table
print("=" * 60)
print("ABLATION: Sampling Strategy (Tau Sensitivity)")
print("=" * 60)
print(f"{'Config':<25} {'FID':>10}")
print("-" * 60)
for name, fid in results['tau_fids'].items():
    marker = " <-- default" if name == 'annealed_0.5_0.9' else ""
    print(f"{name:<25} {fid:>10.2f}{marker}")
print("=" * 60)

print("\n")
print("=" * 60)
print("ABLATION: FID vs Decoding Steps")
print("=" * 60)
header = f"{'Steps':<8}" + "".join(f"{s:>12}" for s in sampler_types)
print(header)
print("-" * 60)
for i, step_val in enumerate(results['step_counts']):
    row = f"{step_val:<8}"
    for s in sampler_types:
        row += f"{results['fid_vs_steps'][s][i]:>12.2f}"
    print(row)
print("=" * 60)

In [ ]:
# Cell 9.10: Save All Results
# Save results dict
import json

# Convert non-serializable values
save_results = {}
for k, v in results.items():
    if isinstance(v, (list, dict, float, int, str)):
        save_results[k] = v
    elif isinstance(v, np.floating):
        save_results[k] = float(v)

with open(f"{config['project_dir']}/results.json", 'w') as fp:
    json.dump(save_results, fp, indent=2)

print(f"All results saved to {config['project_dir']}/results.json")
print(f"All figures saved to {config['project_dir']}/figures/")
print("\nFigures generated:")
for fname in sorted(os.listdir(f"{config['project_dir']}/figures/")):
    print(f"  {fname}")

print("\n" + "="*60)
print("EXPERIMENT COMPLETE!")
print("="*60)
print("You can now use these figures and results in your report.")
print("All checkpoints are saved on Google Drive for reproducibility.")